# 09 — Combine batches without losing schema meaning

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-09")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
a=spark.createDataFrame([(1,'Asha'),(2,'Ben')],'id INT,name STRING')
b=spark.createDataFrame([('Carla',3,'Brazil'),('Ben',2,'UK')],'name STRING,id INT,country STRING')

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Align by column name

Create combined using unionByName with missing columns allowed.

**Expected result:** Four rows, correctly aligned columns.

In [ ]:
combined=None  # TODO: unionByName

**Hint 1:** A positional union can confuse name and ID when columns arrive in a different order.

**Hint 2:** allowMissingColumns supplies null for the absent country.

**Hint 3:** a.unionByName(b, allowMissingColumns=True).

### Check

In [ ]:
check_frame(combined,['id','name','country'],[(1,'Asha',None),(2,'Ben',None),(3,'Carla','Brazil'),(2,'Ben','UK')])

## Task 2: Compare UNION ALL and UNION

Project id/name into view batch_names. Create all_rows with UNION ALL against itself; unique_rows with UNION against itself.

**Expected result:** 8 all rows versus3 unique id/name rows.

In [ ]:
all_rows=None
unique_rows=None  # TODO: SQL set operations

**Hint 1:** UNION ALL keeps duplicates; SQL UNION removes duplicate full rows.

**Hint 2:** Compare only id/name so Ben rows become identical.

**Hint 3:** SELECT id,name FROM ... UNION [ALL] SELECT id,name FROM ... .

### Check

In [ ]:
assert all_rows is not None and all_rows.count()==8
check_frame(unique_rows,['id','name'],[(1,'Asha'),(2,'Ben'),(3,'Carla')])

## Task 3: Retain the most complete record per ID

Create resolved choosing nonnull country before null country for each ID.

**Expected result:** Ben retains UK; three records.

In [ ]:
resolved=None  # TODO: deterministic window ordering

**Hint 1:** Deduplicating a union by ID requires a rule for conflicting payloads.

**Hint 2:** The fixture has only one nonnull country per ID, so completeness is sufficient here.

**Hint 3:** row_number over id ordered by country descending/nulls last; keep rn1.

### Check

In [ ]:
check_frame(resolved,['id','name','country'],[(1,'Asha',None),(2,'Ben','UK'),(3,'Carla','Brazil')])

## More practice

Add incompatible numeric types and normalize before union; explain why DISTINCT is not a change-data merge.

## Common mistakes

unionByName does not guarantee uniqueness. Full-row distinct is different from per-business-key deduplication.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()